# Chapter 25: Hyperbolic Geometry

**Source orientation.** Sections 25.1-25.6; printed pages 483-504; PDF pages 505-526. The source is used for structure and terminology only; the prose, computations, and visuals here are original.

**Chapter question.** How do projective and complex tools give two coordinated models of the same hyperbolic plane?

The chapter narrows the general Cayley-Klein machinery to one nondegenerate absolute: the unit circle. Inside the circle is the material hyperbolic plane. The boundary carries ideal data. The exterior is not a second physical world, but it is valuable algebraic bookkeeping: projective lines continue, intersections do not disappear, and boundary cross-ratios keep measuring angles and distances.

This notebook follows that route computationally. We first mark the staging ground, then compare the Beltrami-Klein and Poincare disk pictures, then use boundary transformations and CP1/Mobius maps to explain why angle and distance formulas are controlled by the unit circle.


## Computational Translation Guide

| Source idea | Computational object in this notebook | What must be checked |
| --- | --- | --- |
| The absolute conic is the unit circle | Homogeneous quadratic form `x^2 + y^2 - z^2 = 0` | Interior values are negative, boundary values vanish, exterior values are positive |
| Hyperbolic lines in the Beltrami-Klein model | Chords with endpoints on the unit circle | Projective joins and intersections still make sense outside the disk |
| Hyperbolic lines in the Poincare disk | Circular arcs orthogonal to the unit circle, with the same boundary endpoints as the Klein chord | Arc points map to one Klein chord under the model map |
| Hyperbolic transformations | Disk-preserving projective boundary actions; in CP1, Mobius or anti-Mobius maps preserving the unit circle | Boundary radius stays one, interior radius stays below one, and hyperbolic distance is invariant |
| Angles | Boundary cross-ratio formula and Euclidean arc intersection angle in the Poincare model | The two angle computations agree numerically |
| Distances | `abs(log(cross_ratio(P,Q;X,Y)))` on the Poincare geodesic through `P,Q` | Cross-ratio distance agrees with the standard Poincare distance formula |

**Library routing.** Matplotlib is used for durable 2D model diagrams and proof panels; Plotly is used for the CP1 transformation lab because a moving Mobius parameter is more inspectable as HTML; NumPy carries the disk geometry; SymPy verifies the one-dimensional distance identity exactly; the course `utils.artifacts`, `utils.projective`, and `utils.conics` helpers keep paths, CSV ledgers, and homogeneous-coordinate checks book-local.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-25-hyperbolic-geometry"
ARTIFACT_ROOT


In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import sympy as sp
from matplotlib.patches import Circle

from utils.artifacts import artifact_path, assert_artifacts, display_artifact, image_stats, save_json, save_table
from utils.conics import evaluate_conic
from utils.projective import affine, cross_ratio, hpoint, incidence, join, meet

plt.ioff()

FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
TABLES = ARTIFACT_ROOT / "tables"
CHECKS = ARTIFACT_ROOT / "checks"
for folder in [FIGURES, HTML, TABLES, CHECKS]:
    folder.mkdir(parents=True, exist_ok=True)

SOURCE_SPAN = "sections 25.1-25.6; printed pages 483-504; PDF pages 505-526"

def rel(path):
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()

def save_figure(fig, filename):
    path = artifact_path(ARTIFACT_ROOT, "figures", filename)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path

def add_unit_disk(ax, title=None):
    ax.add_patch(Circle((0, 0), 1.0, facecolor="#f7fbff", edgecolor="#1f2937", lw=1.8, zorder=0))
    theta = np.linspace(0, 2*np.pi, 360)
    ax.plot(np.cos(theta), np.sin(theta), color="#111827", lw=1.8)
    ax.axhline(0, color="#d1d5db", lw=0.7)
    ax.axvline(0, color="#d1d5db", lw=0.7)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=11, pad=8)

def xy(z):
    z = np.asarray(z)
    return np.real(z), np.imag(z)

def boundary(theta):
    return np.exp(1j * theta)

def klein_from_poincare(z):
    z = np.asarray(z, dtype=complex)
    return 2 * z / (1 + np.abs(z) ** 2)

def poincare_from_klein(w):
    w = np.asarray(w, dtype=complex)
    r2 = np.abs(w) ** 2
    denom = 1 + np.sqrt(np.maximum(0, 1 - r2))
    return np.where(np.abs(w) < 1e-14, 0j, w / denom)

def disk_auto(z, a=0.0j, theta=0.0):
    z = np.asarray(z, dtype=complex)
    return np.exp(1j * theta) * (z - a) / (1 - np.conjugate(a) * z)

def disk_auto_inv(w, a=0.0j, theta=0.0):
    w = np.asarray(w, dtype=complex)
    u = np.exp(-1j * theta) * w
    return (u + a) / (1 + np.conjugate(a) * u)

def disk_anti_auto(z, a=0.0j, theta=0.0):
    return disk_auto(np.conjugate(z), a=a, theta=theta)

def geodesic_boundary_endpoints(p, q):
    p = complex(p)
    q = complex(q)
    if abs(p - q) < 1e-12:
        raise ValueError("A geodesic needs two distinct disk points")
    q0 = disk_auto(q, a=p)
    direction = q0 / abs(q0)
    x = disk_auto_inv(-direction, a=p)
    y = disk_auto_inv(direction, a=p)
    return complex(x), complex(y)

def orthogonal_circle(a, b):
    a = complex(a)
    b = complex(b)
    mat = np.array([[a.real, a.imag], [b.real, b.imag]], dtype=float)
    if abs(np.linalg.det(mat)) < 1e-10:
        return None, math.inf
    center_xy = np.linalg.solve(mat, np.array([1.0, 1.0]))
    c = complex(center_xy[0], center_xy[1])
    radius = math.sqrt(max(0.0, abs(c) ** 2 - 1.0))
    return c, radius

def arc_for_boundary_pair(a, b, n=240):
    a = complex(a)
    b = complex(b)
    c, radius = orthogonal_circle(a, b)
    if c is None:
        return np.linspace(a, b, n)
    t1 = math.atan2((a - c).imag, (a - c).real)
    t2 = math.atan2((b - c).imag, (b - c).real)
    delta_pos = (t2 - t1) % (2 * np.pi)
    delta_neg = -((t1 - t2) % (2 * np.pi))
    candidates = []
    for delta in [delta_pos, delta_neg]:
        ts = t1 + np.linspace(0, delta, n)
        z = c + radius * np.exp(1j * ts)
        candidates.append(z)
    candidates.sort(key=lambda z: (np.nanmax(np.abs(z)), np.nanmean(np.abs(z))))
    return candidates[0]

def geodesic_arc(p, q, n=240):
    a, b = geodesic_boundary_endpoints(p, q)
    return arc_for_boundary_pair(a, b, n=n), (a, b)

def poincare_distance(p, q):
    p = complex(p)
    q = complex(q)
    u = abs((p - q) / (1 - np.conjugate(p) * q))
    u = min(max(float(u), 0.0), 1.0 - 1e-15)
    return 2 * math.atanh(u)

def cross_ratio_distance(p, q):
    x, y = geodesic_boundary_endpoints(p, q)
    cr = cross_ratio(p, q, x, y)
    return abs(math.log(abs(cr))), cr, x, y

def tangent_at_point_on_geodesic(point, endpoints):
    point = complex(point)
    a, b = endpoints
    c, radius = orthogonal_circle(a, b)
    if c is None:
        v = b - a
    else:
        v = 1j * (point - c)
    return v / abs(v)

def oriented_angle_between(v, w):
    angle = abs(np.angle(v / w))
    if angle > np.pi:
        angle = 2*np.pi - angle
    return float(angle)

def boundary_angle_from_cross_ratio(pair_l, pair_m):
    candidates = []
    for al, bl in [pair_l, pair_l[::-1]]:
        for am, bm in [pair_m, pair_m[::-1]]:
            cr = cross_ratio(al, bl, am, bm)
            if abs(cr.imag) < 1e-8 and cr.real < 0:
                angle = 2 * math.atan(math.sqrt(max(0.0, -cr.real)))
                if angle > np.pi:
                    angle = 2*np.pi - angle
                candidates.append((angle, cr, (al, bl), (am, bm)))
    if not candidates:
        raise RuntimeError("Could not orient boundary endpoints to get a negative cross-ratio")
    candidates.sort(key=lambda item: item[0])
    return candidates[0]

def collinearity_residual(points):
    pts = np.array([[z.real, z.imag, 1.0] for z in points], dtype=float)
    _, s, _ = np.linalg.svd(pts)
    return float(s[-1])


## Visual Storyboard

This implementation uses six source-specific inspection targets.

1. **Hyperbolic staging ground.** The unit conic separates material points, ideal boundary points, and exterior projective bookkeeping. The invariant is the sign of `x^2 + y^2 - z^2`.
2. **Klein/Poincare dictionary.** The same boundary endpoints represent a straight chord in Beltrami-Klein and an orthogonal circular arc in Poincare. The invariant is that mapped arc samples become collinear with the Klein chord.
3. **Moving a point to the center.** A disk automorphism sends a chosen interior point to the origin while keeping the boundary circle fixed. The invariant is boundary radius and hyperbolic distance.
4. **Angles from boundary data.** The boundary cross-ratio angle formula matches the Euclidean angle at the Poincare arc intersection. The invariant is the angle residual.
5. **CP1 transformation lab.** A Plotly slider shows Mobius transformations of disk geodesics and the induced projective action on the boundary.
6. **Distance heatmap.** Cross-ratio distance in the Poincare disk agrees with the standard formula and grows rapidly near the ideal boundary.


In [ ]:
storyboard = {
    "chapter goal": "Coordinate the Beltrami-Klein and Poincare disk models through boundary projective data, CP1 transformations, and hyperbolic angle/distance formulas.",
    "source span read": SOURCE_SPAN,
    "concept inventory": [
        "unit disk absolute as the nondegenerate Cayley-Klein conic",
        "material interior H, ideal boundary, and exterior projective bookkeeping",
        "hyperbolic transformations determined by boundary action",
        "Beltrami-Klein chords versus Poincare orthogonal arcs",
        "RP2/CP1 identification on the finite affine chart",
        "Mobius and anti-Mobius disk transformations preserving the unit circle",
        "angle from boundary cross-ratio and conformality in the Poincare disk",
        "distance as the logarithm of a cross-ratio without the Klein factor one half",
    ],
    "library routing table": [
        {"concept": "staging ground and model comparison", "representation": "2D conic/chord/arc diagrams", "library": "Matplotlib + NumPy", "why": "static labeled disk diagrams make the model distinction inspectable"},
        {"concept": "boundary and Mobius transformation", "representation": "parameterized disk-geodesic lab", "library": "Plotly", "why": "the transformation parameter is best inspected interactively in standalone HTML"},
        {"concept": "cross-ratio, incidence, exact one-axis formula", "representation": "numeric and symbolic checks", "library": "utils.projective + SymPy", "why": "homogeneous coordinates and exact algebra expose the invariants"},
        {"concept": "validation ledgers", "representation": "CSV/JSON", "library": "course artifact helpers", "why": "tables make model translations and residuals reproducible"},
    ],
    "visual sequence": [
        {"order": 1, "concept": "hyperbolic staging ground", "artifact": "figures/staging-ground-boundary-zones.png", "inspection target": "which objects are material and which are bookkeeping", "validation": "unit conic sign classification"},
        {"order": 2, "concept": "Klein/Poincare geodesic dictionary", "artifact": "figures/klein-poincare-geodesic-dictionary.png", "inspection target": "same boundary endpoints, different line renderings", "validation": "Poincare arc samples map to Klein chords"},
        {"order": 3, "concept": "hyperbolic transformation of boundary points", "artifact": "figures/boundary-transformation-to-center.png", "inspection target": "the point moves to the origin while boundary stays fixed", "validation": "disk automorphism radius and distance invariance"},
        {"order": 4, "concept": "angle preservation panel", "artifact": "figures/angle-boundary-cross-ratio-panel.png", "inspection target": "cross-ratio angle equals Euclidean arc angle", "validation": "angle residual"},
        {"order": 5, "concept": "CP1 transformations", "artifact": "html/cp1-mobius-disk-lab.html", "inspection target": "Mobius images of geodesics remain geodesics", "validation": "boundary and interior radius checks"},
        {"order": 6, "concept": "distance cross-ratio heatmap", "artifact": "figures/distance-cross-ratio-heatmap.png", "inspection target": "distance blows up near the ideal boundary", "validation": "cross-ratio versus standard distance residuals"},
    ],
    "artifact plan": {
        "figures": [
            "staging-ground-boundary-zones.png",
            "klein-poincare-geodesic-dictionary.png",
            "boundary-transformation-to-center.png",
            "angle-boundary-cross-ratio-panel.png",
            "distance-cross-ratio-heatmap.png",
        ],
        "html": ["cp1-mobius-disk-lab.html"],
        "tables": ["model-translation-guide.csv", "distance-sample-checks.csv"],
        "checks": ["storyboard.json", "visual-checks.json", "final-sanity.json"],
    },
    "computational checks": [
        "unit conic sign classification",
        "projective join/meet incidence for exterior intersection bookkeeping",
        "Klein/Poincare roundtrip residual",
        "Poincare arc samples mapped to Klein chord collinearity residual",
        "Mobius boundary radius, center move, and distance invariance residuals",
        "boundary cross-ratio angle residual",
        "cross-ratio distance residual and exact SymPy one-axis identity",
        "artifact existence, nonzero size, and nonblank raster statistics",
    ],
    "proof-visualization strategy": "Use sign zones, chord/arc correspondence, transformation-to-center, and formula residuals as proof-state views rather than copied source proofs.",
    "implementation notes": "All outputs are book-local under artifacts/chapter-25-hyperbolic-geometry; shared utilities are read but not edited.",
    "risks": ["Only representative transformation families are visualized; the full group proof remains a scaffold of invariants."],
    "acceptance criteria": ["Notebook executes with nbclient", "Chapter 25 artifacts are present and nonblank", "final-sanity.json records small residuals"],
}

storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
storyboard_path


## 1. The Staging Ground: Interior, Boundary, Exterior

A hyperbolic construction lives inside the unit disk, but the projective coordinate system sees more. A point inside the unit conic has negative value under `x^2 + y^2 - z^2`; a boundary point has value zero; an exterior point has positive value. That sign test is the simplest computational version of the chapter's distinction between material points and algebraic extensions.

The figure below also shows two material chord segments that do not meet inside the disk. Their full projective lines still meet outside the disk, which is why projective bookkeeping removes case distinctions that an intrinsic-only description would have to keep.


In [ ]:
unit_conic = np.diag([1.0, 1.0, -1.0])
inside = hpoint(0.28, -0.18)
bound = hpoint(1.0, 0.0)
outside = hpoint(1.18, 0.52)
classification_values = {
    "inside_material": evaluate_conic(unit_conic, inside),
    "boundary_ideal": evaluate_conic(unit_conic, bound),
    "outside_projective_bookkeeping": evaluate_conic(unit_conic, outside),
}

a1, a2 = boundary(np.deg2rad(130)), boundary(np.deg2rad(238))
b1, b2 = boundary(np.deg2rad(-38)), boundary(np.deg2rad(72))
line_a = join(hpoint(a1.real, a1.imag), hpoint(a2.real, a2.imag))
line_b = join(hpoint(b1.real, b1.imag), hpoint(b2.real, b2.imag))
projective_intersection = meet(line_a, line_b)
intersection_xy = affine(projective_intersection)
line_incidence_ok = incidence(projective_intersection, line_a) and incidence(projective_intersection, line_b)

fig, ax = plt.subplots(figsize=(7.1, 6.1))
add_unit_disk(ax, "Unit conic as hyperbolic staging ground")
ax.add_patch(Circle((0, 0), 1.0, facecolor="#dbeafe", edgecolor="none", alpha=0.35, zorder=-2))
ax.text(-0.22, 0.03, "material\ninterior H", color="#1d4ed8", fontsize=10, ha="center")
ax.text(0.72, 0.70, "ideal\nboundary", color="#111827", fontsize=9, ha="center")
ax.text(1.08, -1.10, "exterior bookkeeping", color="#7c2d12", fontsize=9, ha="center")

for endpoints, color, label in [((a1, a2), "#0f766e", "material chord segments"), ((b1, b2), "#9333ea", None)]:
    p, q = endpoints
    t = np.linspace(-0.65, 1.65, 160)
    extended = p + t * (q - p)
    ax.plot(*xy(extended), color=color, lw=1.0, ls="--", alpha=0.55)
    segment = np.linspace(p, q, 80)
    ax.plot(*xy(segment), color=color, lw=3.0, label=label)
    ax.scatter([p.real, q.real], [p.imag, q.imag], color=color, s=26, zorder=4)

ax.scatter([inside[0], bound[0], outside[0]], [inside[1], bound[1], outside[1]],
           color=["#2563eb", "#111827", "#c2410c"], s=[42, 42, 54], zorder=5)
ax.scatter([intersection_xy[0]], [intersection_xy[1]], marker="x", s=90, color="#b91c1c", zorder=6)
ax.annotate("projective intersection\noutside H", xy=intersection_xy, xytext=(0.28, 1.15),
            arrowprops={"arrowstyle": "->", "color": "#b91c1c"}, fontsize=9, color="#7f1d1d")
ax.legend(loc="lower left", frameon=False, fontsize=9)

staging_path = save_figure(fig, "staging-ground-boundary-zones.png")
staging_checks = {
    "classification_values": classification_values,
    "inside_negative": classification_values["inside_material"] < 0,
    "boundary_zero_abs": abs(classification_values["boundary_ideal"]),
    "outside_positive": classification_values["outside_projective_bookkeeping"] > 0,
    "projective_intersection_radius": float(np.linalg.norm(intersection_xy)),
    "projective_intersection_outside_disk": float(np.linalg.norm(intersection_xy)) > 1,
    "line_incidence_ok": bool(line_incidence_ok),
}
staging_path, staging_checks


In [ ]:
display_artifact(staging_path, width=760)


## 2. Beltrami-Klein Chords and Poincare Arcs

The two disk models use the same ideal boundary but draw their lines differently. A Beltrami-Klein line is a straight chord. The matching Poincare line is the unique circular arc through the same ideal endpoints that meets the unit circle orthogonally.

The model map used here is radial:

\[
w = \frac{2z}{1+|z|^2}, \qquad
z = \frac{w}{1+\sqrt{1-|w|^2}}.
\]

Here `z` is the Poincare coordinate and `w` is the Beltrami-Klein coordinate. The validation samples points on Poincare arcs, maps them to Klein coordinates, and checks that the samples are collinear with the corresponding chord endpoints.


In [ ]:
p_z = 0.23 + 0.18j
q_points = [0.72 + 0.12j, -0.50 + 0.50j, -0.18 - 0.64j]
colors = ["#2563eb", "#d97706", "#059669"]

fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.2))
add_unit_disk(axes[0], "Beltrami-Klein: geodesics are chords")
add_unit_disk(axes[1], "Poincare: geodesics are orthogonal arcs")

klein_collinearity_residuals = []
for q, color in zip(q_points, colors):
    arc, endpoints = geodesic_arc(p_z, q)
    a, b = endpoints
    chord = np.linspace(a, b, 160)
    axes[0].plot(*xy(chord), color=color, lw=2.4)
    axes[1].plot(*xy(arc), color=color, lw=2.4)
    axes[0].scatter([a.real, b.real], [a.imag, b.imag], color=color, s=24)
    axes[1].scatter([a.real, b.real], [a.imag, b.imag], color=color, s=24)
    mapped = klein_from_poincare(arc[30:-30:20])
    residual = collinearity_residual([a, b, *mapped])
    klein_collinearity_residuals.append(residual)

p_w = klein_from_poincare(p_z)
q_w = [klein_from_poincare(q) for q in q_points]
axes[1].scatter([p_z.real], [p_z.imag], color="#111827", s=54, zorder=5, label="Poincare point z")
axes[0].scatter([p_w.real], [p_w.imag], color="#111827", s=54, zorder=5, label="Klein point w")
for q, qw, color in zip(q_points, q_w, colors):
    axes[1].scatter([q.real], [q.imag], color=color, s=40, marker="s", zorder=5)
    axes[0].scatter([qw.real], [qw.imag], color=color, s=40, marker="s", zorder=5)
axes[0].legend(frameon=False, loc="lower left", fontsize=9)
axes[1].legend(frameon=False, loc="lower left", fontsize=9)
fig.suptitle("Same ideal endpoints, different rendering of hyperbolic lines", fontsize=13, y=1.02)

kp_path = save_figure(fig, "klein-poincare-geodesic-dictionary.png")
roundtrip_samples = np.array([0j, 0.2+0.1j, -0.45+0.25j, 0.1-0.72j])
roundtrip = poincare_from_klein(klein_from_poincare(roundtrip_samples))
kp_checks = {
    "max_klein_collinearity_residual": float(max(klein_collinearity_residuals)),
    "max_roundtrip_residual": float(np.max(np.abs(roundtrip - roundtrip_samples))),
    "poincare_point_radius": float(abs(p_z)),
    "klein_point_radius": float(abs(p_w)),
}
kp_path, kp_checks


In [ ]:
display_artifact(kp_path, width=900)


## 3. Hyperbolic Transformations: Boundary Action and a Center Move

A disk-preserving transformation is determined by its action on boundary data. In the Poincare disk, the orientation-preserving transformations are Mobius maps of the form

\[
z \mapsto e^{i\theta}\frac{z-a}{1-\bar a z}, \qquad |a|<1.
\]

The chosen parameter `a` is also the point sent to the origin. This realizes the chapter's useful normal form: move a point and incident lines to the center, where angle calculations are simple, while the unit circle remains the same ideal boundary.


In [ ]:
move_point = 0.34 + 0.22j
rotation = 0.35
probe_qs = [0.75 + 0.07j, -0.45 + 0.55j]
boundary_anchors = [boundary(t) for t in np.deg2rad([20, 145, 275])]

fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.2))
add_unit_disk(axes[0], "Before: chosen point and boundary anchors")
add_unit_disk(axes[1], "After: Mobius map sends the point to the center")
for q, color in zip(probe_qs, ["#2563eb", "#d97706"]):
    arc, _ = geodesic_arc(move_point, q)
    image_arc = disk_auto(arc, a=move_point, theta=rotation)
    axes[0].plot(*xy(arc), color=color, lw=2.3)
    axes[1].plot(*xy(image_arc), color=color, lw=2.3)
    axes[0].scatter([q.real], [q.imag], color=color, s=38)
    image_q = disk_auto(q, a=move_point, theta=rotation)
    axes[1].scatter([image_q.real], [image_q.imag], color=color, s=38)

for idx, z in enumerate(boundary_anchors, start=1):
    image = disk_auto(z, a=move_point, theta=rotation)
    axes[0].scatter([z.real], [z.imag], color="#111827", s=36, zorder=5)
    axes[1].scatter([image.real], [image.imag], color="#111827", s=36, zorder=5)
    axes[0].text(z.real*1.08, z.imag*1.08, f"A{idx}", ha="center", va="center", fontsize=9)
    axes[1].text(image.real*1.08, image.imag*1.08, f"A{idx}'", ha="center", va="center", fontsize=9)

image_move_point = disk_auto(move_point, a=move_point, theta=rotation)
axes[0].scatter([move_point.real], [move_point.imag], color="#dc2626", s=64, zorder=6, label="p")
axes[1].scatter([image_move_point.real], [image_move_point.imag], color="#dc2626", s=64, zorder=6, label="tau(p)")
axes[0].legend(frameon=False, loc="lower left", fontsize=9)
axes[1].legend(frameon=False, loc="lower left", fontsize=9)
fig.suptitle("Boundary-preserving Mobius maps give hyperbolic normal forms", fontsize=13, y=1.02)

transform_path = save_figure(fig, "boundary-transformation-to-center.png")
pre_p, pre_q = move_point, 0.12 - 0.58j
post_p = disk_auto(pre_p, a=move_point, theta=rotation)
post_q = disk_auto(pre_q, a=move_point, theta=rotation)
anti_probe = disk_anti_auto(np.array([0.2+0.1j, -0.4+0.2j]), a=0.18-0.1j, theta=0.2)
transformation_checks = {
    "chosen_point_image_abs": float(abs(image_move_point)),
    "max_boundary_radius_error": float(max(abs(abs(disk_auto(z, a=move_point, theta=rotation)) - 1) for z in boundary_anchors)),
    "sample_distance_before": float(poincare_distance(pre_p, pre_q)),
    "sample_distance_after": float(poincare_distance(post_p, post_q)),
    "distance_invariance_residual": float(abs(poincare_distance(pre_p, pre_q) - poincare_distance(post_p, post_q))),
    "anti_mobius_interior_max_radius": float(np.max(np.abs(anti_probe))),
}
transform_path, transformation_checks


In [ ]:
display_artifact(transform_path, width=900)


## 4. Angles from Boundary Cross-Ratios

For two oriented hyperbolic lines with boundary endpoints `A_l,B_l` and `A_m,B_m`, the chapter's angle formula can be tested directly:

\[
\alpha = 2\arctan\sqrt{-(A_l,B_l;A_m,B_m)_H}.
\]

The Poincare model is conformal, so the same angle should be the ordinary Euclidean intersection angle of the two circular arcs. The panel below shows both computations on one example. The tangent arrows are Euclidean, but the value they produce is hyperbolic because the arcs meet the boundary orthogonally and the Mobius normal form preserves circle angles.


In [ ]:
angle_point = 0.18 + 0.16j
angle_q1 = -0.50 + 0.58j
angle_q2 = 0.70 + 0.38j
arc1, pair1 = geodesic_arc(angle_point, angle_q1)
arc2, pair2 = geodesic_arc(angle_point, angle_q2)
formula_angle, angle_cr, oriented_pair1, oriented_pair2 = boundary_angle_from_cross_ratio(pair1, pair2)
tan1 = tangent_at_point_on_geodesic(angle_point, oriented_pair1)
tan2 = tangent_at_point_on_geodesic(angle_point, oriented_pair2)
euclidean_angle = oriented_angle_between(tan1, tan2)
angle_residual = abs(formula_angle - euclidean_angle)

fig, ax = plt.subplots(figsize=(7.2, 6.2))
add_unit_disk(ax, "Angle from boundary data in the Poincare disk")
ax.plot(*xy(arc1), color="#2563eb", lw=2.6, label="line l")
ax.plot(*xy(arc2), color="#d97706", lw=2.6, label="line m")
for z, label, color in [(oriented_pair1[0], "A_l", "#2563eb"), (oriented_pair1[1], "B_l", "#2563eb"),
                        (oriented_pair2[0], "A_m", "#d97706"), (oriented_pair2[1], "B_m", "#d97706")]:
    ax.scatter([z.real], [z.imag], color=color, s=32, zorder=5)
    ax.text(1.10*z.real, 1.10*z.imag, label, fontsize=9, ha="center", va="center")
ax.scatter([angle_point.real], [angle_point.imag], color="#111827", s=58, zorder=6)
scale = 0.28
for tangent, color in [(tan1, "#1d4ed8"), (tan2, "#b45309")]:
    start = angle_point - 0.5*scale*tangent
    delta = scale*tangent
    ax.arrow(start.real, start.imag, delta.real, delta.imag, width=0.006, head_width=0.035,
             color=color, length_includes_head=True, zorder=7)
ax.text(-1.26, -1.22,
        f"cross-ratio = {angle_cr.real:.6f}\nangle formula = {math.degrees(formula_angle):.3f} deg\nEuclidean arc angle = {math.degrees(euclidean_angle):.3f} deg",
        fontsize=9, color="#111827", bbox={"facecolor": "white", "edgecolor": "#e5e7eb", "alpha": 0.92})
ax.legend(frameon=False, loc="upper left", fontsize=9)

angle_path = save_figure(fig, "angle-boundary-cross-ratio-panel.png")
angle_checks = {
    "boundary_cross_ratio_real": float(angle_cr.real),
    "boundary_cross_ratio_imag_abs": float(abs(angle_cr.imag)),
    "formula_angle_radians": float(formula_angle),
    "euclidean_arc_angle_radians": float(euclidean_angle),
    "angle_residual": float(angle_residual),
}
angle_path, angle_checks


In [ ]:
display_artifact(angle_path, width=760)


## 5. CP1 Transformation Lab

The finite affine chart of `RP^2` identifies `(x,y,1)^T` with `x + iy` in `CP^1`. On the unit circle, cross-ratio coordinates give the same projective boundary action in both languages. The lab below moves a Mobius parameter `a` and redraws several Poincare geodesics. The inspection target is simple: the boundary remains the unit circle, and transformed geodesics remain orthogonal-circle arcs inside it.

The static notebook view links the HTML artifact. Opening it gives a slider through several disk automorphisms.


In [ ]:
base_angles = np.linspace(0, np.pi, 7, endpoint=False)
base_geodesics = [np.linspace(-np.exp(1j*t), np.exp(1j*t), 180) for t in base_angles]
unit = np.exp(1j*np.linspace(0, 2*np.pi, 300))
params = np.linspace(0.0, 0.55, 8)
frames = []
initial_data = None
palette = ["#2563eb", "#7c3aed", "#d97706", "#059669", "#dc2626", "#0891b2", "#4b5563"]
for k, radius in enumerate(params):
    a = radius * np.exp(0.55j)
    theta = 0.35
    traces = [go.Scatter(x=unit.real, y=unit.imag, mode="lines", line={"color": "#111827", "width": 2}, name="unit boundary")]
    for idx, geo in enumerate(base_geodesics):
        image = disk_auto(geo, a=a, theta=theta)
        traces.append(go.Scatter(x=image.real, y=image.imag, mode="lines",
                                 line={"color": palette[idx % len(palette)], "width": 2},
                                 name=f"geodesic {idx+1}", showlegend=(k == 0)))
    marker = disk_auto(np.array([a]), a=a, theta=theta)[0]
    traces.append(go.Scatter(x=[marker.real], y=[marker.imag], mode="markers",
                             marker={"color": "#dc2626", "size": 9}, name="image of a"))
    if k == 0:
        initial_data = traces
    frames.append(go.Frame(data=traces, name=f"a={radius:.2f}"))

fig = go.Figure(data=initial_data, frames=frames)
fig.update_layout(
    title="CP1/Mobius disk automorphisms preserve the hyperbolic boundary",
    width=760,
    height=660,
    xaxis={"range": [-1.18, 1.18], "scaleanchor": "y", "showgrid": False, "zeroline": False},
    yaxis={"range": [-1.18, 1.18], "showgrid": False, "zeroline": False},
    margin={"l": 20, "r": 20, "t": 58, "b": 20},
    sliders=[{
        "steps": [{"method": "animate", "label": f"{r:.2f}", "args": [[f"a={r:.2f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}]} for r in params],
        "currentvalue": {"prefix": "|a| = "},
    }],
    updatemenus=[{"type": "buttons", "showactive": False, "x": 0.02, "y": 1.06,
                  "buttons": [{"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 450, "redraw": True}, "fromcurrent": True}]}]}],
)
cp1_html_path = artifact_path(ARTIFACT_ROOT, "html", "cp1-mobius-disk-lab.html")
fig.write_html(cp1_html_path, include_plotlyjs=True, full_html=True)

boundary_samples = np.exp(1j*np.linspace(0, 2*np.pi, 25, endpoint=False))
interior_samples = np.array([0.0+0.0j, 0.3+0.2j, -0.4+0.1j, 0.1-0.5j])
lab_a = 0.42*np.exp(0.55j)
cp1_checks = {
    "max_boundary_radius_error": float(np.max(np.abs(np.abs(disk_auto(boundary_samples, a=lab_a, theta=0.35)) - 1))),
    "max_interior_image_radius": float(np.max(np.abs(disk_auto(interior_samples, a=lab_a, theta=0.35)))),
    "anti_mobius_boundary_radius_error": float(np.max(np.abs(np.abs(disk_anti_auto(boundary_samples, a=lab_a, theta=0.35)) - 1))),
    "html_file_size": int(cp1_html_path.stat().st_size),
}
cp1_html_path, cp1_checks


In [ ]:
display_artifact(cp1_html_path, width="100%", height=620)


## 6. Distances: Cross-Ratio Heatmap

For Poincare disk points `P,Q`, let `X,Y` be the ideal endpoints of their Poincare geodesic. The chapter's distance formula is

\[
d(P,Q)=\left|\log(P,Q;X,Y)\right|.
\]

This differs from the Beltrami-Klein normalization because the Poincare-to-Klein radial map squares the relevant real-axis cross-ratio. The heatmap fixes a base point and colors every other disk point by distance. The color growth near the boundary is the visual version of ideal points being infinitely far away.


In [ ]:
base = 0.25 - 0.18j
n = 260
xs = np.linspace(-0.985, 0.985, n)
ys = np.linspace(-0.985, 0.985, n)
xx, yy = np.meshgrid(xs, ys)
zz = xx + 1j*yy
mask = np.abs(zz) < 0.985
with np.errstate(divide="ignore", invalid="ignore"):
    uu = np.abs((zz - base) / (1 - np.conjugate(base) * zz))
    uu = np.clip(uu, 0, 1 - 1e-12)
    dist_grid = 2 * np.arctanh(uu)
dist_grid[~mask] = np.nan

fig, ax = plt.subplots(figsize=(7.2, 6.3))
image = ax.imshow(dist_grid, origin="lower", extent=[xs.min(), xs.max(), ys.min(), ys.max()], cmap="viridis", vmin=0, vmax=5.0)
add_unit_disk(ax, "Poincare distance from a base point")
ax.scatter([base.real], [base.imag], color="#ef4444", s=54, zorder=5, label="base point")
for radius in [0.25, 0.5, 0.75, 0.9]:
    circle = radius*np.exp(1j*np.linspace(0, 2*np.pi, 220))
    ax.plot(circle.real, circle.imag, color="white", lw=0.7, alpha=0.35)
q_demo = -0.44 + 0.48j
arc_demo, endpoints_demo = geodesic_arc(base, q_demo)
ax.plot(*xy(arc_demo), color="#fbbf24", lw=2.2, label="sample geodesic")
ax.scatter([q_demo.real], [q_demo.imag], color="#fbbf24", edgecolor="#111827", s=48, zorder=5)
cbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("hyperbolic distance", fontsize=9)
ax.legend(frameon=False, loc="lower left", fontsize=9)

distance_path = save_figure(fig, "distance-cross-ratio-heatmap.png")

sample_points = [0.0+0.0j, -0.35+0.12j, 0.52+0.35j, -0.16-0.66j, q_demo]
distance_rows = []
max_distance_residual = 0.0
for idx, q in enumerate(sample_points, start=1):
    d_cr, cr, x, y = cross_ratio_distance(base, q)
    d_std = poincare_distance(base, q)
    residual = abs(d_cr - d_std)
    max_distance_residual = max(max_distance_residual, residual)
    distance_rows.append({
        "sample": idx,
        "base": f"{base.real:.6f}{base.imag:+.6f}i",
        "point": f"{q.real:.6f}{q.imag:+.6f}i",
        "endpoint_X_radius": abs(x),
        "endpoint_Y_radius": abs(y),
        "cross_ratio_abs": abs(cr),
        "distance_cross_ratio": d_cr,
        "distance_standard_formula": d_std,
        "absolute_residual": residual,
    })

distance_table_path = save_table(distance_rows, ARTIFACT_ROOT, "tables", "distance-sample-checks.csv")
distance_checks = {
    "max_distance_residual": float(max_distance_residual),
    "base_radius": float(abs(base)),
    "heatmap_finite_min": float(np.nanmin(dist_grid)),
    "heatmap_finite_max_clipped_view": float(np.nanmax(np.minimum(dist_grid, 5.0))),
    "sample_endpoint_radius_error": float(max(abs(row["endpoint_X_radius"] - 1) + abs(row["endpoint_Y_radius"] - 1) for row in distance_rows)),
}
distance_path, distance_table_path, distance_checks


In [ ]:
display_artifact(distance_path, width=760)
display_artifact(distance_table_path)


## Proof and Invariant Scaffold

The chapter repeatedly reduces hard-looking statements to a model where the computation is easy:

- Move a point to the origin using a boundary-preserving transformation.
- Use boundary endpoints because a projective action on the conic is controlled by cross-ratio data.
- Translate a Poincare arc into a Klein chord when linear incidence is the useful representation.
- Translate a Klein chord into a Poincare arc when conformal angle measurement is the useful representation.

The next cell records two algebraic sanity checks behind those reductions. First, homogeneous joins and meets confirm that the exterior intersection in the staging-ground panel really lies on both projective lines. Second, SymPy verifies the real-axis identity that explains why the Poincare distance formula loses the Beltrami-Klein factor `1/2`: under `w=2z/(1+z^2)`, the endpoint ratio is squared.


In [ ]:
p_sym, q_sym = sp.symbols("p q", real=True)
wp_expr = 2*p_sym/(1+p_sym**2)
wq_expr = 2*q_sym/(1+q_sym**2)
klein_axis_cr = ((wp_expr - 1)*(wq_expr + 1))/((wp_expr + 1)*(wq_expr - 1))
poincare_axis_cr_squared = (((p_sym - 1)*(q_sym + 1))/((p_sym + 1)*(q_sym - 1)))**2
axis_identity = sp.simplify(klein_axis_cr - poincare_axis_cr_squared)

translation_rows = [
    {"chapter_concept": "absolute conic", "coordinate_model": "diag(1,1,-1)", "artifact": rel(staging_path), "invariant": "sign classifies inside/boundary/outside"},
    {"chapter_concept": "Klein geodesic", "coordinate_model": "chord between ideal endpoints", "artifact": rel(kp_path), "invariant": "mapped Poincare samples are collinear"},
    {"chapter_concept": "Poincare geodesic", "coordinate_model": "circle orthogonal to unit circle", "artifact": rel(kp_path), "invariant": "same ideal endpoints as Klein chord"},
    {"chapter_concept": "hyperbolic transformation", "coordinate_model": "Mobius disk automorphism", "artifact": rel(transform_path), "invariant": "boundary radius and distance are preserved"},
    {"chapter_concept": "angle", "coordinate_model": "boundary cross-ratio", "artifact": rel(angle_path), "invariant": "equals Euclidean Poincare arc angle"},
    {"chapter_concept": "distance", "coordinate_model": "log cross-ratio", "artifact": rel(distance_path), "invariant": "equals standard Poincare distance"},
]
translation_table_path = save_table(translation_rows, ARTIFACT_ROOT, "tables", "model-translation-guide.csv")

proof_checks = {
    "projective_intersection_incidence_ok": bool(line_incidence_ok),
    "axis_distance_identity_simplifies_to_zero": str(axis_identity),
    "axis_distance_identity_ok": axis_identity == 0,
}
translation_table_path, proof_checks


In [ ]:
display_artifact(translation_table_path)


## Applied Lab: Design a Boundary-Preserving Experiment

A useful experiment for this chapter is not to draw another disk picture, but to ask which invariant you expect before you run the transformation.

1. Pick two disk points `p` and `q`.
2. Pick a Mobius parameter `a` with `|a|<1` and an angle `theta`.
3. Transform both points with `disk_auto`.
4. Compare the before/after hyperbolic distance and the before/after Euclidean distance.

The hyperbolic distance should remain fixed. The Euclidean distance usually changes. That difference is the practical reason the disk picture should not be read as ordinary Euclidean metric geometry, even though the Poincare model preserves angles.


In [ ]:
lab_p, lab_q = -0.18 + 0.34j, 0.58 - 0.14j
lab_a, lab_theta = 0.31 - 0.18j, -0.42
lab_p_img = disk_auto(lab_p, a=lab_a, theta=lab_theta)
lab_q_img = disk_auto(lab_q, a=lab_a, theta=lab_theta)
lab_result = {
    "euclidean_distance_before": float(abs(lab_p - lab_q)),
    "euclidean_distance_after": float(abs(lab_p_img - lab_q_img)),
    "hyperbolic_distance_before": float(poincare_distance(lab_p, lab_q)),
    "hyperbolic_distance_after": float(poincare_distance(lab_p_img, lab_q_img)),
    "hyperbolic_distance_residual": float(abs(poincare_distance(lab_p, lab_q) - poincare_distance(lab_p_img, lab_q_img))),
}
lab_result


## Artifact Gallery

These direct links keep the notebook readable before execution and give the audits explicit artifact references.

![Hyperbolic staging ground](../../artifacts/chapter-25-hyperbolic-geometry/figures/staging-ground-boundary-zones.png)

![Klein and Poincare geodesic dictionary](../../artifacts/chapter-25-hyperbolic-geometry/figures/klein-poincare-geodesic-dictionary.png)

![Boundary transformation to center](../../artifacts/chapter-25-hyperbolic-geometry/figures/boundary-transformation-to-center.png)

![Angle boundary cross-ratio panel](../../artifacts/chapter-25-hyperbolic-geometry/figures/angle-boundary-cross-ratio-panel.png)

![Distance cross-ratio heatmap](../../artifacts/chapter-25-hyperbolic-geometry/figures/distance-cross-ratio-heatmap.png)

Open the [CP1 Mobius disk lab](../../artifacts/chapter-25-hyperbolic-geometry/html/cp1-mobius-disk-lab.html), the [model translation guide](../../artifacts/chapter-25-hyperbolic-geometry/tables/model-translation-guide.csv), and the [distance sample checks](../../artifacts/chapter-25-hyperbolic-geometry/tables/distance-sample-checks.csv) for the supporting ledgers.


## Final Sanity Checks

The final cell checks the notebook's core claims rather than just checking that files were written. A failure here should point to a mathematical mistake: a non-invariant Mobius distance, a broken chord/arc dictionary, a bad boundary angle, stale artifacts, or a blank rendered figure.


In [ ]:
artifact_paths = [
    staging_path,
    kp_path,
    transform_path,
    angle_path,
    distance_path,
    cp1_html_path,
    translation_table_path,
    distance_table_path,
    storyboard_path,
]
assert_artifacts(artifact_paths, min_size=256)

raster_stats = [image_stats(path) for path in [staging_path, kp_path, transform_path, angle_path, distance_path]]
raster_stats = [{**stats, "path": rel(stats["path"])} for stats in raster_stats]

all_checks = {
    "chapter": 25,
    "source_span": SOURCE_SPAN,
    "staging": staging_checks,
    "klein_poincare": kp_checks,
    "transformation": transformation_checks,
    "angle": angle_checks,
    "cp1_lab": cp1_checks,
    "distance": distance_checks,
    "proof": proof_checks,
    "applied_lab": lab_result,
    "artifacts": [rel(path) for path in artifact_paths],
    "raster_artifacts": raster_stats,
}

assert all_checks["staging"]["inside_negative"]
assert all_checks["staging"]["boundary_zero_abs"] < 1e-12
assert all_checks["staging"]["outside_positive"]
assert all_checks["staging"]["projective_intersection_outside_disk"]
assert all_checks["staging"]["line_incidence_ok"]
assert all_checks["klein_poincare"]["max_klein_collinearity_residual"] < 1e-10
assert all_checks["klein_poincare"]["max_roundtrip_residual"] < 1e-12
assert all_checks["transformation"]["chosen_point_image_abs"] < 1e-12
assert all_checks["transformation"]["max_boundary_radius_error"] < 1e-12
assert all_checks["transformation"]["distance_invariance_residual"] < 1e-12
assert all_checks["angle"]["boundary_cross_ratio_imag_abs"] < 1e-8
assert all_checks["angle"]["angle_residual"] < 1e-10
assert all_checks["cp1_lab"]["max_boundary_radius_error"] < 1e-12
assert all_checks["cp1_lab"]["max_interior_image_radius"] < 1.0
assert all_checks["distance"]["max_distance_residual"] < 1e-10
assert all_checks["distance"]["sample_endpoint_radius_error"] < 1e-10
assert all_checks["proof"]["axis_distance_identity_ok"]
assert all_checks["applied_lab"]["hyperbolic_distance_residual"] < 1e-12
for stats in all_checks["raster_artifacts"]:
    assert stats["width"] >= 200 and stats["height"] >= 150
    assert stats["pixel_std"] > 1.0

visual_checks = {
    "all_files_exist": all(Path(path).exists() for path in artifact_paths),
    "cross_ratio_error": all_checks["distance"]["max_distance_residual"],
    "numeric_checks": {
        "angle_residual": all_checks["angle"]["angle_residual"],
        "complex_cross_ratio_residual": all_checks["distance"]["max_distance_residual"],
        "mobius_distance_residual": all_checks["transformation"]["distance_invariance_residual"],
        "klein_poincare_collinearity_residual": all_checks["klein_poincare"]["max_klein_collinearity_residual"],
    },
    "raster_artifacts": all_checks["raster_artifacts"],
    "html_artifact": rel(cp1_html_path),
    "table_artifacts": [rel(translation_table_path), rel(distance_table_path)],
    "visual_count": 8,
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final_sanity = {
    **all_checks,
    "artifacts": [rel(path) for path in [*artifact_paths, visual_checks_path]],
    "visual_checks": rel(visual_checks_path),
    "notebook_executed": True,
}
final_sanity_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity


## Takeaways

- The unit circle is not just a drawing boundary; it is the absolute conic that controls which projective points are material, ideal, or exterior bookkeeping.
- Beltrami-Klein and Poincare models share boundary endpoints. Chords make incidence linear; orthogonal arcs make angle measurement conformal.
- Hyperbolic transformations are most visible through their boundary action. In CP1, the corresponding maps are Mobius or anti-Mobius transformations preserving the unit circle.
- The Poincare distance formula is a boundary cross-ratio formula. The heatmap and table show the same invariant numerically, while the SymPy identity explains the normalization change from the Klein formula.
